# 🤖 AI Stock Analyzer — Real Portfolio Analysis

This notebook analyzes your **real holdings** from `Current_Holding_Apr26.csv` using:
- Live price data via **Yahoo Finance** (`yfinance`)
- Technical indicators: **SMA, RSI, Bollinger Bands**
- Machine Learning signal prediction (**Random Forest**)
- SEC **10-K annual report** parsing and summarization
- **GPU acceleration** via PyTorch (auto-detected)

> **⚙️ Where to see GPU usage:** Run Cell 2 below — it prints a full GPU/CPU device report.  
> **📂 Holdings file:** `data/Current_Holding_Apr26.csv` (edit path in Cell 3 if needed)

## Cell 1 — Install Dependencies

Run once. Skip if packages are already installed.

In [ ]:
# Install required packages (uncomment and run once)
# !pip install yfinance pandas numpy matplotlib seaborn plotly \
#              requests beautifulsoup4 lxml scikit-learn tqdm ipywidgets
#
# For GPU support (CUDA 12.1):
# !pip install torch --index-url https://download.pytorch.org/whl/cu121
#
# For CPU-only PyTorch:
# !pip install torch --index-url https://download.pytorch.org/whl/cpu
print('Dependencies ready.')

## Cell 2 — 🖥️ GPU / Device Status

**This cell shows whether analysis will use your GPU or CPU.**

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from gpu_utils import print_device_report, get_model_device

device_info = print_device_report()
DEVICE = get_model_device()
print(f"\n✅ Models will run on: {device_info['device_name']}")

## Cell 3 — 📂 Load Current Holdings

Update `HOLDINGS_PATH` to point to your CSV file.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from stock_analyzer import HoldingsLoader

# ── UPDATE THIS PATH IF NEEDED ─────────────────────────────────────────────
HOLDINGS_PATH = Path('data/Current_Holding_Apr26.csv')
# ──────────────────────────────────────────────────────────────────────────

loader = HoldingsLoader()
holdings = loader.load(str(HOLDINGS_PATH))

print(f'✅ Loaded {len(holdings)} holdings from: {HOLDINGS_PATH}')
print()

# Display full table
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
holdings.style \
    .format({
        'Cost_Basis_Per_Share': '${:,.2f}',
        'Current_Price': '${:,.2f}',
        'Market_Value': '${:,.2f}',
        'Unrealized_Gain_Loss': '${:+,.2f}',
        'Unrealized_Gain_Loss_Pct': '{:+.2f}%',
    }) \
    .background_gradient(
        subset=['Unrealized_Gain_Loss_Pct'],
        cmap='RdYlGn',
        vmin=-30,
        vmax=100
    ) \
    .set_table_styles([{'selector': 'table', 'props': [('width', '100%')]}])

## Cell 4 — 💵 Portfolio Summary

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

total_cost   = (holdings['Cost_Basis_Per_Share'] * holdings['Quantity']).sum()
total_value  = holdings['Market_Value'].sum()
total_pnl    = holdings['Unrealized_Gain_Loss'].sum()
total_pnl_pct = total_pnl / total_cost * 100 if total_cost else 0

print('=' * 60)
print('  PORTFOLIO SUMMARY')
print('=' * 60)
print(f'  Total Holdings       : {len(holdings)}')
print(f'  Total Cost Basis     : ${total_cost:>14,.2f}')
print(f'  Total Market Value   : ${total_value:>14,.2f}')
print(f'  Unrealized P&L       : ${total_pnl:>+14,.2f}  ({total_pnl_pct:+.2f}%)')
print('=' * 60)

# ── Allocation pie chart ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Portfolio Overview', fontsize=16, fontweight='bold')

# Pie — market value allocation
ax1 = axes[0]
colors = plt.cm.Set3(np.linspace(0, 1, len(holdings)))
ax1.pie(
    holdings['Market_Value'],
    labels=holdings['Symbol'],
    autopct='%1.1f%%',
    colors=colors,
    startangle=140,
)
ax1.set_title('Market Value Allocation')

# Bar — unrealized gain/loss per holding
ax2 = axes[1]
bar_colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in holdings['Unrealized_Gain_Loss']]
bars = ax2.bar(holdings['Symbol'], holdings['Unrealized_Gain_Loss'], color=bar_colors, edgecolor='white')
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_title('Unrealized Gain/Loss per Holding ($)')
ax2.set_xlabel('Symbol')
ax2.set_ylabel('Unrealized P&L ($)')
ax2.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for bar, val in zip(bars, holdings['Unrealized_Gain_Loss']):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + (200 if val >= 0 else -600),
        f'${val:+,.0f}',
        ha='center', va='bottom', fontsize=8, fontweight='bold'
    )

plt.tight_layout()
plt.show()

## Cell 5 — 📈 Live Prices from Yahoo Finance

In [ ]:
from stock_analyzer import MarketDataFetcher

fetcher = MarketDataFetcher()
tickers = holdings['Symbol'].tolist()

print(f'Fetching live prices for {len(tickers)} tickers …')
live_prices = fetcher.fetch_current_prices(tickers)

# Merge with holdings
holdings_live = holdings.merge(live_prices, on='Symbol', how='left')

display_cols = ['Symbol', 'Description', 'Quantity', 'Cost_Basis_Per_Share',
                'Live_Price', 'Day_Change_Pct', 'Market_Value', 'Unrealized_Gain_Loss_Pct']
available = [c for c in display_cols if c in holdings_live.columns]

holdings_live[available].style \
    .format({
        'Cost_Basis_Per_Share': '${:,.2f}',
        'Live_Price': '${:,.2f}',
        'Day_Change_Pct': '{:+.2f}%',
        'Market_Value': '${:,.2f}',
        'Unrealized_Gain_Loss_Pct': '{:+.2f}%',
    }) \
    .background_gradient(subset=['Day_Change_Pct'], cmap='RdYlGn', vmin=-5, vmax=5) \
    .background_gradient(subset=['Unrealized_Gain_Loss_Pct'], cmap='RdYlGn', vmin=-30, vmax=100) \
    .set_table_styles([{'selector': 'table', 'props': [('width', '100%')]}])

## Cell 6 — 📊 Technical Indicators + ML Signals

In [ ]:
from stock_analyzer import TechnicalIndicators, ReturnPredictor

indicators = TechnicalIndicators()
predictor  = ReturnPredictor(device=DEVICE)

results = []

for ticker in tickers:
    print(f'Analyzing {ticker} …', end=' ')
    hist = fetcher.fetch_history(ticker, period='1y')
    if hist.empty:
        print('NO DATA')
        continue

    hist_ind = indicators.compute_all(hist)
    signal   = predictor.fit_predict(hist_ind)
    latest   = hist_ind.iloc[-1]

    results.append({
        'Symbol':    ticker,
        'Close':     latest.get('Close'),
        'RSI_14':    latest.get('RSI_14'),
        'SMA_20':    latest.get('SMA_20'),
        'SMA_50':    latest.get('SMA_50'),
        'BB_Upper':  latest.get('BB_Upper'),
        'BB_Lower':  latest.get('BB_Lower'),
        'ML_Signal': signal or 'N/A',
    })
    print(f'✓  ML Signal: {signal}')

results_df = pd.DataFrame(results)

print()
results_df.style \
    .format({
        'Close':    '${:,.2f}',
        'RSI_14':   '{:.1f}',
        'SMA_20':   '${:,.2f}',
        'SMA_50':   '${:,.2f}',
        'BB_Upper': '${:,.2f}',
        'BB_Lower': '${:,.2f}',
    }, na_rep='N/A') \
    .background_gradient(subset=['RSI_14'], cmap='RdYlGn', vmin=30, vmax=70) \
    .set_table_styles([{'selector': 'table', 'props': [('width', '100%')]}])

## Cell 7 — 📉 Price + Indicator Charts (per stock)

In [ ]:
def plot_stock(ticker: str) -> None:
    hist = fetcher.fetch_history(ticker, period='1y')
    if hist.empty:
        print(f'No data for {ticker}')
        return
    df = indicators.compute_all(hist)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
    fig.suptitle(f'{ticker} — 1-Year Price & Indicators', fontsize=14, fontweight='bold')

    # Price + Bollinger Bands + SMAs
    ax1.plot(df.index, df['Close'],    label='Close', color='steelblue', linewidth=1.5)
    ax1.plot(df.index, df['SMA_20'],   label='SMA 20', color='orange',   linestyle='--', linewidth=1)
    ax1.plot(df.index, df['SMA_50'],   label='SMA 50', color='purple',   linestyle='--', linewidth=1)
    ax1.fill_between(df.index, df['BB_Upper'], df['BB_Lower'], alpha=0.1, color='gray', label='BB Bands')
    ax1.plot(df.index, df['BB_Upper'], color='gray', linestyle=':', linewidth=0.8)
    ax1.plot(df.index, df['BB_Lower'], color='gray', linestyle=':', linewidth=0.8)
    ax1.set_ylabel('Price ($)')
    ax1.legend(loc='upper left', fontsize=8)
    ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax1.grid(True, alpha=0.3)

    # RSI
    ax2.plot(df.index, df['RSI_14'], color='teal', linewidth=1.5, label='RSI 14')
    ax2.axhline(70, color='red',   linestyle='--', linewidth=0.8, label='Overbought (70)')
    ax2.axhline(30, color='green', linestyle='--', linewidth=0.8, label='Oversold (30)')
    ax2.fill_between(df.index, df['RSI_14'], 70, where=df['RSI_14'] >= 70, alpha=0.2, color='red')
    ax2.fill_between(df.index, df['RSI_14'], 30, where=df['RSI_14'] <= 30, alpha=0.2, color='green')
    ax2.set_ylabel('RSI')
    ax2.set_ylim(0, 100)
    ax2.legend(loc='upper left', fontsize=8)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Plot all holdings
for t in tickers:
    plot_stock(t)

## Cell 8 — 📄 SEC 10-K Analysis

Fetches and summarizes the most recent annual 10-K filing for each holding from SEC EDGAR.

> ⚠️ This makes live HTTP requests to `sec.gov`. Set `TICKERS_FOR_10K` to limit which tickers are fetched.

In [ ]:
from sec_10k_analyzer import SEC10KAnalyzer

# ── Limit to specific tickers to avoid long waits ─────────────────────────
TICKERS_FOR_10K = ['MU', 'NVDA', 'AAPL']   # ← change as needed
# ──────────────────────────────────────────────────────────────────────────

sec = SEC10KAnalyzer(cache=True)

for ticker in TICKERS_FOR_10K:
    print(f'\nFetching 10-K for {ticker} …')
    report = sec.get_10k_summary(ticker)
    print(report)

## Cell 9 — 📦 Full Pipeline (one-shot)

Runs the complete `PortfolioAnalyzer` pipeline (matches the CLI `start.bat`).

In [ ]:
from stock_analyzer import PortfolioAnalyzer

analyzer = PortfolioAnalyzer(
    holdings_path=str(HOLDINGS_PATH),
    run_10k=False,   # Set True to also fetch 10-K filings
)
analyzer.run()